### Losigtic regresson
### decision tree classifier
### random forest classifier
### tubulate orqali jadval tuzish
# Imtihonga tayyorgarlik

`bank-full.csv` datasetida preprocessing → encoding → scaling → train/test → **4 ta model alohida bo'limlarda** natija → oxirida accuracy jadvali.

In [6]:
import pandas as pd

# Windows pathda \U unicode escape xatoligi bo'lmasligi uchun raw string ishlatamiz
df = pd.read_csv(r"C:\Users\PC\MLFoundation\Sherzod_dev\2-oy_Model_Building\Data\bank-full.csv")
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 1) Preprocessing: bo'sh qiymatlarni to'ldirish

In [7]:
def datalarni_toldirsin(df):
    df = df.copy()
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == "object":
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].mean())
    return df


df = datalarni_toldirsin(df)
print("Null count:", df.isnull().sum().sum())

Null count: 0


## 2) Encoding (categorical -> numeric)

In [8]:
from sklearn.preprocessing import LabelEncoder

def encoding_qilsin(df):
    df = df.copy()
    label_encoder = LabelEncoder()
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = label_encoder.fit_transform(df[col])
    return df


df = encoding_qilsin(df)
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,4,1,2,0,2143,1,0,2,5,8,261,1,-1,0,3,0
1,44,9,2,1,0,29,1,0,2,5,8,151,1,-1,0,3,0
2,33,2,1,1,0,2,1,1,2,5,8,76,1,-1,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,8,92,1,-1,0,3,0
4,33,11,2,3,0,1,0,0,2,5,8,198,1,-1,0,3,0


## 3) Train/Testga ajratish va Scaling

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (36168, 16) X_test: (9043, 16)


## 4) To'rtta model (har biri alohida)

Quyida har bir model **alohida bo'lim**: `fit` → `predict` → **accuracy** → **classification_report**.
Eng oxirida barcha modellar **accuracy** bo'yicha bir jadvalda qisqacha solishtiriladi.

### 4.1 Logistic Regression


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

logistic_model = LogisticRegression(max_iter=2000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)

y_pred_logistic = logistic_model.predict(X_test_scaled)
acc_logistic = accuracy_score(y_test, y_pred_logistic)
print("=== Logistic Regression ===")
print("Accuracy:", round(acc_logistic, 4))
print(classification_report(y_test, y_pred_logistic))


=== Logistic Regression ===
Accuracy: 0.8914
              precision    recall  f1-score   support

           0       0.91      0.98      0.94      7985
           1       0.59      0.23      0.33      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.60      0.63      9043
weighted avg       0.87      0.89      0.87      9043



### 4.2 Decision Tree Classifier


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

y_pred_tree = decision_tree_model.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)
print("=== Decision Tree Classifier ===")
print("Accuracy:", round(acc_tree, 4))
print(classification_report(y_test, y_pred_tree))


=== Decision Tree Classifier ===
Accuracy: 0.8769
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      7985
           1       0.47      0.48      0.48      1058

    accuracy                           0.88      9043
   macro avg       0.70      0.70      0.70      9043
weighted avg       0.88      0.88      0.88      9043



### 4.3 Random Forest Classifier


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

random_forest_model = RandomForestClassifier(n_estimators=200, random_state=42)
random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("=== Random Forest Classifier ===")
print("Accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf))


=== Random Forest Classifier ===
Accuracy: 0.9053
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      7985
           1       0.65      0.41      0.50      1058

    accuracy                           0.91      9043
   macro avg       0.79      0.69      0.72      9043
weighted avg       0.89      0.91      0.90      9043



### 4.4 Linear Regression (binary: threshold = 0.5)

Regressor chiqishi uzluksiz; klassifikatsiya uchun `>= 0.5` bilan 0/1 ga aylantiramiz.


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

y_pred_linear_raw = linear_model.predict(X_test_scaled)
y_pred_linear = (y_pred_linear_raw >= 0.5).astype(int)
acc_linear = accuracy_score(y_test, y_pred_linear)
print("=== Linear Regression (threshold=0.5) ===")
print("Accuracy:", round(acc_linear, 4))
print(classification_report(y_test, y_pred_linear))


=== Linear Regression (threshold=0.5) ===
Accuracy: 0.8874
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      7985
           1       0.60      0.11      0.19      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.55      0.57      9043
weighted avg       0.86      0.89      0.85      9043



### 5) Accuracy bo'yicha solishtirish (jadval)


In [14]:
results = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Decision Tree Classifier",
            "Random Forest Classifier",
            "Linear Regression (as classifier)",
        ],
        "Accuracy": [acc_logistic, acc_tree, acc_rf, acc_linear],
    }
).sort_values("Accuracy", ascending=False)

try:
    from tabulate import tabulate

    print(
        tabulate(
            results.reset_index(drop=True),
            headers="keys",
            tablefmt="github",
            floatfmt=".4f",
        )
    )
except ImportError:
    print("(tabulate o'rnatilmagan — yuqoridagi DataFrame yetarli)")

results


|    | Model                             |   Accuracy |
|----|-----------------------------------|------------|
|  0 | Random Forest Classifier          |     0.9053 |
|  1 | Logistic Regression               |     0.8914 |
|  2 | Linear Regression (as classifier) |     0.8874 |
|  3 | Decision Tree Classifier          |     0.8769 |


,Model,Accuracy
2,Random Forest Classifier,0.905341
0,Logistic Regression,0.891408
3,Linear Regression (as classifier),0.887427
1,Decision Tree Classifier,0.876921


### 6) Final qo'shimcha: RandomForestClassifier (oxirgi tekshiruv)

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_final = RandomForestClassifier(n_estimators=300, random_state=42)
rf_final.fit(X_train, y_train)

y_pred_rf_final = rf_final.predict(X_test)
acc_rf_final = accuracy_score(y_test, y_pred_rf_final)

print("=== Final RandomForestClassifier ===")
print("Accuracy:", round(acc_rf_final, 4))
print(classification_report(y_test, y_pred_rf_final))

=== Final RandomForestClassifier ===
Accuracy: 0.9079
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      7985
           1       0.67      0.42      0.52      1058

    accuracy                           0.91      9043
   macro avg       0.80      0.70      0.73      9043
weighted avg       0.90      0.91      0.90      9043

